# Feature Engineering

**Tujuan**: Merekayasa fitur baru dari `clean_data_after_eda.csv` dan `price_data.csv` untuk meningkatkan kemampuan prediktif model churn.

**Ide awal dari Estelle**: selisih harga off-peak antara bulan Desember dan Januari tahun sebelumnya mungkin merupakan fitur penting.

**Kerangka kerja yang diikuti**:

* Import & load data
* Fitur: selisih harga off-peak Des vs Jan (ide Estelle)
* Fitur tambahan kreatif (date, consumption, margin, price volatility)
* Encoding variabel kategorikal
* Menghapus kolom yang tidak relevan / redundan
* Menggabungkan seluruh fitur menjadi satu dataset final

## Import packages

In [2]:
import pandas as pd
import numpy as np

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## Load data

In [4]:
df = pd.read_csv('/content/drive/MyDrive/Forage/Data Science - BCG /clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

print(df.shape)
df.head(5)

(14606, 44)


,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,9.084737e-04,2.086294,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000e+00,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000e+00,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000011,2.896760e-06,4.860000e-10,0.000000,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0


## Fitur ide Estelle: selisih harga off-peak Desember vs Januari

kita pakai `price_data.csv` (data harga mentah per bulan) untuk menghitung selisih harga off-peak (variabel & fixed) antara bulan terakhir (Desember) dan bulan pertama (Januari) tahun sebelumnya, per pelanggan.

In [5]:
price_df = pd.read_csv('/content/drive/MyDrive/Forage/Data Science - BCG /price_data (1).csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head(5)

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [6]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [7]:
# Uji cepat: apakah fitur ini terlihat berbeda antara pelanggan churn vs retention?
temp = pd.merge(diff, df[['id', 'churn']], on='id')
temp.groupby('churn')[['offpeak_diff_dec_january_energy', 'offpeak_diff_dec_january_power']].mean()

,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
churn,,
0,-0.004561,0.277313
1,-0.004605,0.289483


**Insight:**

Kita akan cek nanti apakah rata-rata selisih ini berbeda cukup jauh antara kelompok churn vs retention. Tapi terlepas dari itu, fitur ini tetap kita simpan dulu karena model (bukan hanya rata-rata) yang menulai daya prediksinya secara lebiih rigor (ketat/penuh ketelitian) termasuk kombinasi dengan fitur yg lain.

## Fitur tambahan kreatif

Selain idenya Estelle, berikut ini adalah fitur-fitur yang saya buat berdasarkan pemahaman bisnis dan eksplorasi data sebelumnya.

### Volatilitas harga di periode lain (peak & mid-peak), bukan hanya off-peak

Disini, ide Estelle berfokus pada off-peak. Sekarang kita perluas lagi logika yang sama untuk periode peak dan mid-peak, karena volatilitas harga di periode manapun berpotensi memengaruhi persepsi customer.

In [8]:
monthly_price_full = price_df.groupby(['id', 'price_date']).agg({
    'price_off_peak_var': 'mean', 'price_peak_var': 'mean', 'price_mid_peak_var': 'mean',
    'price_off_peak_fix': 'mean', 'price_peak_fix': 'mean', 'price_mid_peak_fix': 'mean'
}).reset_index()

jan_full = monthly_price_full.groupby('id').first().reset_index()
dec_full = monthly_price_full.groupby('id').last().reset_index()

price_cols = ['price_off_peak_var', 'price_peak_var', 'price_mid_peak_var',
              'price_off_peak_fix', 'price_peak_fix', 'price_mid_peak_fix']

diff_full = jan_full[['id']].copy()
for col in price_cols:
    diff_full[f'diff_dec_january_{col}'] = dec_full[col].values - jan_full[col].values

diff_full.head()

,id,diff_dec_january_price_off_peak_var,diff_dec_january_price_peak_var,diff_dec_january_price_mid_peak_var,diff_dec_january_price_off_peak_fix,diff_dec_january_price_peak_fix,diff_dec_january_price_mid_peak_fix
0,0002203ffbb812588b632b9e628cc38d,-0.006192,-0.002302,0.003487,0.162916,0.097749,0.065166
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.000000,0.000000,0.177779,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,0.000000,0.000000,1.500000,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,-0.005120,0.000763,0.162916,0.097749,0.065166
4,00114d74e963e47177db89bc70108537,-0.003994,0.000000,0.000000,-0.000001,0.000000,0.000000


### Volatilitas harga keseluruhan sepanjang tahun (bukan hanya titik Desember vs Januari)

Seliih dua titik waktu yaitu (Des-Jan) bisa jadi hanya menangkap "noise" satu bulan. Kita tambahkan juga rentang (max-min) dan standar deviasi harga sepanjang tahun per pelanggannya, perlakuan ini menangkap seberapa volatil harga yang dialami customer secara keseluruhan.

In [9]:
price_volatility = price_df.groupby('id')[price_cols].agg(['max', 'min', 'std']).reset_index()
price_volatility.columns = ['id'] + [f'{col}_{stat}' for col, stat in price_volatility.columns[1:]]

for col in ['price_off_peak_var', 'price_peak_var', 'price_mid_peak_var']:
    price_volatility[f'{col}_range'] = price_volatility[f'{col}_max'] - price_volatility[f'{col}_min']

price_volatility = price_volatility[['id'] + [c for c in price_volatility.columns if c.endswith('_std') or c.endswith('_range')]]
price_volatility.head()

,id,price_off_peak_var_std,price_peak_var_std,price_mid_peak_var_std,price_off_peak_fix_std,price_peak_fix_std,price_mid_peak_fix_std,price_off_peak_var_range,price_peak_var_range,price_mid_peak_var_range
0,0002203ffbb812588b632b9e628cc38d,0.003976,0.001989,0.001368,6.341481e-02,0.038049,0.025366,0.008161,0.004169,0.003541
1,0004351ebdd665e6ee664792efc4fd13,0.002197,0.000000,0.000000,8.753223e-02,0.000000,0.000000,0.004462,0.000000,0.000000
2,0010bcc39e42b3c2131ed2ce55246e3c,0.026008,0.000000,0.000000,7.723930e-01,0.000000,0.000000,0.054905,0.000000,0.000000
3,0010ee3855fdea87602a5b7aba8e42de,0.005049,0.002580,0.000403,8.507958e-02,0.051048,0.034032,0.010018,0.005120,0.000817
4,00114d74e963e47177db89bc70108537,0.002202,0.000000,0.000000,5.908392e-07,0.000000,0.000000,0.004462,0.000000,0.000000


### Fitur berbasis tanggal (ekstraksi & durasi)

Kolom tanggal mentah tidak langsung bermanfaat untuk model. Kita ekstrak komponen yang lebih bermakna seperti: usia kontrak (tenture), bulan aktivasi (musiman), dan sisa waktu hingga kontrak berakhir/perpanjangan.

In [10]:
reference_date = pd.Timestamp('2016-01-01')  # akhir periode observasi data

df['tenure_days'] = (reference_date - df['date_activ']).dt.days
df['months_to_end'] = (df['date_end'] - reference_date).dt.days / 30
df['months_since_modif_prod'] = (reference_date - df['date_modif_prod']).dt.days / 30
df['months_to_renewal'] = (df['date_renewal'] - reference_date).dt.days / 30
df['activ_month'] = df['date_activ'].dt.month  # musiman - apakah bulan aktivasi tertentu lebih rentan churn?

df[['tenure_days', 'months_to_end', 'months_since_modif_prod', 'months_to_renewal', 'activ_month']].describe()

,tenure_days,months_to_end,months_since_modif_prod,months_to_renewal,activ_month
count,14606.000000,14606.000000,14606.000000,14606.000000,14606.000000
mean,1798.670615,6.962232,36.449324,-5.456967,6.558880
std,589.317924,3.564454,30.749099,3.947033,3.514151
min,487.000000,0.900000,-0.933333,-30.633333,1.000000
25%,1352.000000,3.908333,6.633333,-8.633333,3.000000
50%,1764.000000,7.100000,30.866667,-5.266667,7.000000
75%,2177.000000,10.133333,65.600000,-2.133333,10.000000
max,4620.000000,17.633333,154.000000,0.900000,12.000000


### Fitur konsumsi & margin gabungan

Menggabungkan kolom terkait untuk menangkap sinyal yang lebih banyak daripada kolom invidual:
* Total konsumsi energi (listrik + gas)
* Rasio konsumsi gas terhadap listrik (0 jika bukan pelanggan gas)
* Selisih antara konsumsi aktual vs perkiraan (forecast), indikasi seberapa akurasi prediksi PowerCo terhadap customer ini.
* Margin bersih per produk aktif (efisiensi margin per unit produk)

*Net margin (atau net profit margin) adalah persentase dari total pendapatan yang tersisa sebagai keuntungan murni setelah dikurangi seluruh biaya, termasuk harga pokok penjualan (HPP), pajak, dan bunga. Rasio ini mengukur seberapa efektif perusahaan mengelola operasionalnya untuk menghasilkan laba*

In [11]:
df['total_consumption'] = df['cons_12m'] + df['cons_gas_12m']
df['gas_to_elec_ratio'] = np.where(df['cons_12m'] > 0, df['cons_gas_12m'] / df['cons_12m'], 0)
df['forecast_actual_cons_diff'] = df['forecast_cons_12m'] - df['cons_12m']
df['net_margin_per_product'] = np.where(df['nb_prod_act'] > 0, df['net_margin'] / df['nb_prod_act'], df['net_margin'])
df['has_gas_flag'] = (df['has_gas'] == 't').astype(int)

df[['total_consumption', 'gas_to_elec_ratio', 'forecast_actual_cons_diff', 'net_margin_per_product', 'has_gas_flag']].describe()

,total_consumption,gas_to_elec_ratio,forecast_actual_cons_diff,net_margin_per_product,has_gas_flag
count,1.460600e+04,14606.000000,1.460600e+04,14606.000000,14606.000000
mean,1.873127e+05,11.249958,-1.573517e+05,161.513347,0.181501
std,6.683769e+05,954.832626,5.730070e+05,225.147389,0.385446
min,0.000000e+00,0.000000,-6.207019e+06,0.000000,0.000000
25%,6.222250e+03,0.000000,-3.746204e+04,41.745000,0.000000
50%,1.633950e+04,0.000000,-1.260576e+04,95.075000,0.000000
75%,5.082175e+04,0.000000,-4.961927e+03,205.770000,0.000000
max,6.799539e+06,104748.750000,3.534240e+03,10203.500000,1.000000


## Encoding variabel kategorikal

`channel_sales` dan `origin_map` adalah kategorikal dengan beberapa kategori langka dan nilai "MISSING" kita:
* Kelompokkan kategori dengan frekuensi sangat rendah (<1% data) menjadi "other" agar tidak menghasilkan kolom dummy yang nyaris kosong (rentan overfitting)
* One-hot encode hasilnya

In [12]:
def group_rare_categories(series, threshold=0.01):
    freq = series.value_counts(normalize=True)
    rare = freq[freq < threshold].index
    return series.replace(rare, 'other')

df['channel_sales_grouped'] = group_rare_categories(df['channel_sales'])
df['origin_up_grouped'] = group_rare_categories(df['origin_up'])

print(df['channel_sales_grouped'].value_counts())
print()
print(df['origin_up_grouped'].value_counts())

channel_sales_grouped
foosdfpfkusacimwkcsosbicdxkicaua    6754
MISSING                             3725
lmkebamcaaclubfxadlmueccxoimlema    1843
usilxuppasemubllopkaafesmlibmsdf    1375
ewpakwlliwisiwduibdlfmalxowmwpci     893
other                                 16
Name: count, dtype: int64

origin_up_grouped
lxidpiddsbxsbosboudacockeimpuepw    7097
kamkkxfxxuwbdslkwifmmcsiusiuosws    4294
ldkssxwpmemidmecebumciepifcamkci    3148
other                                 67
Name: count, dtype: int64


In [13]:
df = pd.get_dummies(df, columns=['channel_sales_grouped', 'origin_up_grouped'], drop_first=False)
df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,channel_sales_grouped_MISSING,channel_sales_grouped_ewpakwlliwisiwduibdlfmalxowmwpci,channel_sales_grouped_foosdfpfkusacimwkcsosbicdxkicaua,channel_sales_grouped_lmkebamcaaclubfxadlmueccxoimlema,channel_sales_grouped_other,channel_sales_grouped_usilxuppasemubllopkaafesmlibmsdf,origin_up_grouped_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_grouped_ldkssxwpmemidmecebumciepifcamkci,origin_up_grouped_lxidpiddsbxsbosboudacockeimpuepw,origin_up_grouped_other
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,False,False,True,False,False,False,False,False,True,False
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,True,False,False,False,False,False,True,False,False,False
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,False,False,True,False,False,False,True,False,False,False
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,False,False,False,True,False,False,True,False,False,False
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,True,False,False,False,False,False,True,False,False,False


# Menghapus kolom yang tidak relevan/ redundan

Berikut adalah alasan penghapusan kolom:

* `channel_sales`, `origin_up` --> sudah digantikan dengan versi grouped + one-hot encoded
* `has_gas` --> sudah digantikan `has_gas_flag` (numerik)
* `date activ`,`date end`, `date_modif_prod`,`date_renewal` --> sudah diekstrak menjadi fitur numerik (`tenure_days`,`months_to_end`,dst) karena bentuk tanggal mentah tidak bisa langsung digunakan oleh model
* `var_year_price_off_peak`,`var_year_price_peak`,`var_year_price_mid_peak` --> hampir identik dengan penjumlahan `_var` + `_fix` yang sudah ada secara terpisah, redundan (kolinearitas tinggi)
* `var_6m_price_off_peak`, `var_6m_price_peak`, `var_6m_price_mid_peak` --> sama alasannya dengan yang sebelumnya
* `margin_gross_pow_ele` --> sangat berkorelasi dengan `margin_net_pow_ele` (mengukur hal yang sama), disimpan salah satu untuk menghindari duplikasi sinyal

In [14]:
cols_to_drop = [
    'channel_sales', 'origin_up', 'has_gas',
    'date_activ', 'date_end', 'date_modif_prod', 'date_renewal',
    'var_year_price_off_peak', 'var_year_price_peak', 'var_year_price_mid_peak',
    'var_6m_price_off_peak', 'var_6m_price_peak', 'var_6m_price_mid_peak',
    'margin_gross_pow_ele'
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("Kolom tersisa:", df.shape[1])
df.columns.tolist()

Kolom tersisa: 50


['id',
 'cons_12m',
 'cons_gas_12m',
 'cons_last_month',
 'forecast_cons_12m',
 'forecast_cons_year',
 'forecast_discount_energy',
 'forecast_meter_rent_12m',
 'forecast_price_energy_off_peak',
 'forecast_price_energy_peak',
 'forecast_price_pow_off_peak',
 'imp_cons',
 'margin_net_pow_ele',
 'nb_prod_act',
 'net_margin',
 'num_years_antig',
 'pow_max',
 'var_year_price_off_peak_var',
 'var_year_price_peak_var',
 'var_year_price_mid_peak_var',
 'var_year_price_off_peak_fix',
 'var_year_price_peak_fix',
 'var_year_price_mid_peak_fix',
 'var_6m_price_off_peak_var',
 'var_6m_price_peak_var',
 'var_6m_price_mid_peak_var',
 'var_6m_price_off_peak_fix',
 'var_6m_price_peak_fix',
 'var_6m_price_mid_peak_fix',
 'churn',
 'tenure_days',
 'months_to_end',
 'months_since_modif_prod',
 'months_to_renewal',
 'activ_month',
 'total_consumption',
 'gas_to_elec_ratio',
 'forecast_actual_cons_diff',
 'net_margin_per_product',
 'has_gas_flag',
 'channel_sales_grouped_MISSING',
 'channel_sales_grouped_ew

# Menggabungkan seluruh fiitur menjadi satu dataset final

In [15]:
final_df = df.merge(diff, on='id', how='left') \
             .merge(diff_full, on='id', how='left') \
             .merge(price_volatility, on='id', how='left')

print("Final shape:", final_df.shape)
final_df.head()

Final shape: (14606, 67)


,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,...,diff_dec_january_price_mid_peak_fix,price_off_peak_var_std,price_peak_var_std,price_mid_peak_var_std,price_off_peak_fix_std,price_peak_fix_std,price_mid_peak_fix_std,price_off_peak_var_range,price_peak_var_range,price_mid_peak_var_range
0,24011ae4ebbe3035111d65fa7c15bc57,0,54946,0,0.00,0,0.0,1.78,0.114481,0.098142,...,-16.226389,0.007829,0.005126,0.020983,1.050136,7.039226,4.692817,0.028554,0.018480,0.073873
1,d29c2c54acc38ff3c0614d0a653813dd,4660,0,0,189.95,0,0.0,16.27,0.145711,0.000000,...,0.000000,0.002212,0.024677,0.000000,0.080404,0.000000,0.000000,0.005334,0.085483,0.000000
2,764c75f661154dac3a6c254cd082ea7d,544,0,0,47.96,0,0.0,38.72,0.165794,0.087899,...,0.000000,0.002396,0.000506,0.000000,0.087532,0.000000,0.000000,0.004670,0.001281,0.000000
3,bba03439a292a1e166f80264c16191cb,1584,0,0,240.04,0,0.0,19.83,0.146694,0.000000,...,0.000000,0.002317,0.000000,0.000000,0.080403,0.000000,0.000000,0.004547,0.000000,0.000000
4,149d57cf92fc41cf94415803a877cb4b,4425,0,526,445.75,526,0.0,131.73,0.116900,0.100015,...,0.065166,0.003847,0.001885,0.001588,0.073681,0.044209,0.029473,0.008161,0.004169,0.003541


In [16]:
# Cek missing value setelah merge (misal ada id yang tidak muncul di price_data)
final_df.isnull().sum()[final_df.isnull().sum() > 0]

,0


In [17]:
# Isi missing value hasil merge dengan 0 (mengindikasikan tidak ada data harga tersedia untuk periode itu)
merge_cols = [c for c in final_df.columns if c not in df.columns]
final_df[merge_cols] = final_df[merge_cols].fillna(0)

final_df.isnull().sum().sum()

np.int64(0)

# Menyimpan dataset final untuk tahap modelling

In [18]:
final_df.to_csv('data_for_predictions.csv', index=False)
print("Tersimpan sebagai 'data_for_predictions.csv' dengan", final_df.shape[1], "kolom dan", final_df.shape[0], "baris.")

Tersimpan sebagai 'data_for_predictions.csv' dengan 67 kolom dan 14606 baris.


# Ringkasan Feature Engineering (Summary of Feature Engineering)

Berikut adalah fitur baru yang dibuat:

1. `offpeak_diff_dec_january_energy`,`offpeak_diff_dec_january_power`: (keduanya adalah ide inti Estelle)
2. `diff_dec_january_price_*` (6 kolom): perluasan idenya Estelle ke periode peak & mid-peak
3. `price_*_std`, `price_*_range`: volatilitas harga sepanjang tahun
4. `tenure_days`, `months_to_end`, `months_since_modif_prod`, `months_to_renewal`, `active_month`: fitur berbasis tanggal
5. `total_consumption`, `gas_to_elec_ratio`, `forecast-actual_cons_diff`, `net_margin_per_product`, `has_gas_flag`: fitur konsumsi dan margin gabungan
6. One-hot encoding untuk `channel_sales` dan `origin_up` (kategori langka dikelompokkan menjadi `other`)

**kolom yang dihapus**:
* kolom tanggal mentah
* kolom kategorikal mentah, dan
* beberapa kolom yang secara matematis redundan (kolinearitas tinggi dengan kolom lain)


**Final results**: dataset `data_for_predictions.csv` siap digunakan untuk tahap pemodelan menggunakan **Random forest** yang bertujuan untuk menguji fitur mana yang benar-benar berkontribusi memprediksi churn, termasuk menguji apakah idenya Estelle mengenai selisih harga off-peak Desember-Januari terbukti penting melalui feature importance model.

###